# Generic Auto EAZY for JWST NIRCam

#### To Use
<p>This code takes online catalogs from DAWN & MSAEXP; to change the catalog, change the URL paths in Steps 1 & 2. Run parameters are found under '# DEFINITIONS'. Run all following cells to generate the necessary files, then run this shell command:

 ../src/eazy -p {RUN_NAME}_eazy_full.param</p>

In [16]:
import re
import eazy
import msaexp
import grizli

import numpy as np

from grizli import utils
from pathlib import Path
from datetime import datetime

from astropy.utils.data import download_file
from scipy.spatial import cKDTree

print(f'grizli version: {grizli.__version__}')
print(f'eazy-py version: {eazy.__version__}')
print(f'msaexp version: {msaexp.__version__}')


grizli version: 1.13.2
eazy-py version: 0.8.5
msaexp version: 0.9.12


In [17]:
# --- Step 1: Load MSAEXP emission line catalog ---

version = "v4.4" # Updated September 5, 2025.  Include all public spectra even without redshift / line fits

CACHE_DOWNLOADS = True
URL_PREFIX = "https://zenodo.org/records/15472354/files/"

# --- Load MSAEXP emission line catalog ---
table_url = f"{URL_PREFIX}/dja_msaexp_emission_lines_{version}.csv.gz"
spectra = utils.read_catalog(download_file(table_url, cache=CACHE_DOWNLOADS), format='csv')

spectra

file,srcid,ra,dec,grating,filter,effexptm,nfiles,dataset,msamet,msaid,msacnf,dithn,slitid,root,npix,ndet,wmin,wmax,wmaxsn,sn10,flux10,err10,sn50,flux50,err50,sn90,flux90,err90,xstart,ystart,xsize,ysize,slit_pa,pa_v3,srcypix,profcen,profsig,ctime,version,exptime,contchi2,dof,fullchi2,line_ariii_7138,line_ariii_7138_err,line_ariii_7753,line_ariii_7753_err,line_bra,line_bra_err,line_brb,line_brb_err,line_brd,line_brd_err,line_brg,line_brg_err,line_hb,line_hb_err,line_hd,line_hd_err,line_hei_1083,line_hei_1083_err,line_hei_3889,line_hei_3889_err,line_hei_5877,line_hei_5877_err,line_hei_7065,line_hei_7065_err,line_hei_8446,line_hei_8446_err,line_heii_4687,line_heii_4687_err,line_hg,line_hg_err,line_lya,line_lya_err,line_mgii,line_mgii_err,line_neiii_3867,line_neiii_3867_err,line_neiii_3968,line_neiii_3968_err,line_nev_3346,line_nev_3346_err,line_nevi_3426,line_nevi_3426_err,line_niii_1750,line_niii_1750_err,line_oi_6302,line_oi_6302_err,line_oii,line_oii_7325,line_oii_7325_err,line_oii_err,line_oiii,line_oiii_1663,line_oiii_1663_err,line_oiii_4363,line_oiii_4363_err,line_oiii_4959,line_oiii_4959_err,line_oiii_5007,line_oiii_5007_err,line_oiii_err,line_pa10,line_pa10_err,line_pa8,line_pa8_err,line_pa9,line_pa9_err,line_paa,line_paa_err,line_pab,line_pab_err,line_pad,line_pad_err,line_pag,line_pag_err,line_pfb,line_pfb_err,line_pfd,line_pfd_err,line_pfe,line_pfe_err,line_pfg,line_pfg_err,line_sii,line_sii_err,line_siii_9068,line_siii_9068_err,line_siii_9531,line_siii_9531_err,spl_0,spl_0_err,spl_1,spl_10,spl_10_err,spl_11,spl_11_err,spl_12,spl_12_err,spl_13,spl_13_err,spl_14,spl_14_err,spl_15,spl_15_err,spl_16,spl_16_err,spl_17,spl_17_err,spl_18,spl_18_err,spl_19,spl_19_err,spl_1_err,spl_2,spl_20,spl_20_err,spl_21,spl_21_err,spl_22,spl_22_err,spl_2_err,spl_3,spl_3_err,spl_4,spl_4_err,spl_5,spl_5_err,spl_6,spl_6_err,spl_7,spl_7_err,spl_8,spl_8_err,spl_9,spl_9_err,zline,line_civ_1549,line_civ_1549_err,line_h10,line_h10_err,line_h11,line_h11_err,line_h12,line_h12_err,line_h7,line_h7_err,line_h8,line_h8_err,line_h9,line_h9_err,line_ha,line_ha_err,line_hei_6680,line_hei_6680_err,line_heii_1640,line_heii_1640_err,line_nii_6549,line_nii_6549_err,line_nii_6584,line_nii_6584_err,line_oii_7323,line_oii_7323_err,line_oii_7332,line_oii_7332_err,line_sii_6717,line_sii_6717_err,line_sii_6731,line_sii_6731_err,line_siii_6314,line_siii_6314_err,escale0,escale1,line_ciii_1906,line_ciii_1906_err,line_niv_1487,line_niv_1487_err,line_pah_3p29,line_pah_3p29_err,line_pah_3p40,line_pah_3p40_err,eqw_ariii_7138,eqw_ariii_7753,eqw_bra,eqw_brb,eqw_brd,eqw_brg,eqw_ciii_1906,eqw_civ_1549,eqw_ha_nii,eqw_hb,eqw_hd,eqw_hei_1083,eqw_hei_3889,eqw_hei_5877,eqw_hei_7065,eqw_hei_8446,eqw_heii_1640,eqw_heii_4687,eqw_hg,eqw_lya,eqw_mgii,eqw_neiii_3867,eqw_neiii_3968,eqw_nev_3346,eqw_nevi_3426,eqw_niii_1750,eqw_niv_1487,eqw_oi_6302,eqw_oii,eqw_oii_7325,eqw_oiii,eqw_oiii_1663,eqw_oiii_4363,eqw_oiii_4959,eqw_oiii_5007,eqw_pa10,eqw_pa8,eqw_pa9,eqw_paa,eqw_pab,eqw_pad,eqw_pag,eqw_pfb,eqw_pfd,eqw_pfe,eqw_pfg,eqw_sii,eqw_siii_9068,eqw_siii_9531,line_ha_nii,line_ha_nii_err,eqw_h10,eqw_h11,eqw_h12,eqw_h7,eqw_h8,eqw_h9,eqw_ha,eqw_hei_6680,eqw_nii_6549,eqw_nii_6584,eqw_oii_7323,eqw_oii_7332,eqw_sii_6717,eqw_sii_6731,eqw_siii_6314,sn_line,ztime,line_ci_9850,line_ci_9850_err,line_feii_11128,line_feii_11128_err,line_pii_11886,line_pii_11886_err,line_feii_12570,line_feii_12570_err,eqw_ci_9850,eqw_feii_11128,eqw_pii_11886,eqw_feii_12570,line_feii_16440,line_feii_16440_err,line_feii_16877,line_feii_16877_err,line_brf,line_brf_err,line_feii_17418,line_feii_17418_err,line_bre,line_bre_err,line_feii_18362,line_feii_18362_err,eqw_feii_16440,eqw_feii_16877,eqw_brf,eqw_feii_17418,eqw_bre,eqw_feii_18362,valid,objid,z_best,ztype,z_prism,z_grating,phot_correction,phot_flux_radius,phot_dr,file_phot,id_phot,phot_mag_auto,phot_f090w_tot_1,phot_f090w_etot_1,phot_f115w_tot_1,phot_f115w_etot_1,phot_f150w_tot_1,phot_f150w_etot_1,phot_f200w_tot_1,phot_f200w_etot_1,phot_f277w_tot_1,phot_f277w_etot_1,phot_f

In [18]:
# --- Step 2: Load DAWN photometric catalog ---

# Specify field and URL path
field = 'gds-grizli-v7.0'
url_path = 'https://s3.amazonaws.com/grizli-v2/JwstMosaics/v7'

# Download and read catalog
phot = utils.read_catalog(f'{url_path}/{field}_phot.fits')
phot

id,thresh,npix,tnpix,xmin,xmax,ymin,ymax,x,y,x2_image,y2_image,xy_image,errx2,erry2,errxy,a_image,b_image,theta_image,cxx_image,cyy_image,cxy_image,cflux,flux,cpeak,peak,xcpeak,ycpeak,xpeak,ypeak,flag,x_image,y_image,number,ra,dec,x_world,y_world,flux_iso,fluxerr_iso,area_iso,mag_iso,kron_radius,kron_rcirc,flux_auto,fluxerr_auto,bkg_auto,flag_auto,area_auto,flux_radius_flag,flux_radius_20,flux_radius,flux_radius_90,tot_corr,mag_auto,magerr_auto,flux_aper_0,fluxerr_aper_0,flag_aper_0,bkg_aper_0,mask_aper_0,flux_aper_1,fluxerr_aper_1,flag_aper_1,bkg_aper_1,mask_aper_1,flux_aper_2,fluxerr_aper_2,flag_aper_2,bkg_aper_2,mask_aper_2,flux_aper_3,fluxerr_aper_3,flag_aper_3,bkg_aper_3,mask_aper_3,clearp-f430m_flux_aper_0,clearp-f430m_fluxerr_aper_0,clearp-f430m_flag_aper_0,clearp-f430m_bkg_aper_0,clearp-f430m_mask_aper_0,clearp-f430m_flux_aper_1,clearp-f430m_fluxerr_aper_1,clearp-f430m_flag_aper_1,clearp-f430m_bkg_aper_1,clearp-f430m_mask_aper_1,clearp-f430m_flux_aper_2,clearp-f430m_fluxerr_aper_2,clearp-f430m_flag_aper_2,clearp-f430m_bkg_aper_2,clearp-f430m_mask_aper_2,clearp-f430m_flux_aper_3,clearp-f430m_fluxerr_aper_3,clearp-f430m_flag_aper_3,clearp-f430m_bkg_aper_3,clearp-f430m_mask_aper_3,clearp-f430m_tot_corr,clearp-f480m_flux_aper_0,clearp-f480m_fluxerr_aper_0,clearp-f480m_flag_aper_0,clearp-f480m_bkg_aper_0,clearp-f480m_mask_aper_0,clearp-f480m_flux_aper_1,clearp-f480m_fluxerr_aper_1,clearp-f480m_flag_aper_1,clearp-f480m_bkg_aper_1,clearp-f480m_mask_aper_1,clearp-f480m_flux_aper_2,clearp-f480m_fluxerr_aper_2,clearp-f480m_flag_aper_2,clearp-f480m_bkg_aper_2,clearp-f480m_mask_aper_2,clearp-f480m_flux_aper_3,clearp-f480m_fluxerr_aper_3,clearp-f480m_flag_aper_3,clearp-f480m_bkg_aper_3,clearp-f480m_mask_aper_3,clearp-f480m_tot_corr,f090w-clear_flux_aper_0,f090w-clear_fluxerr_aper_0,f090w-clear_flag_aper_0,f090w-clear_bkg_aper_0,f090w-clear_mask_aper_0,f090w-clear_flux_aper_1,f090w-clear_fluxerr_aper_1,f090w-clear_flag_aper_1,f090w-clear_bkg_aper_1,f090w-clear_mask_aper_1,f090w-clear_flux_aper_2,f090w-clear_fluxerr_aper_2,f090w-clear_flag_aper_2,f090w-clear_bkg_aper_2,f090w-clear_mask_aper_2,f090w-clear_flux_aper_3,f090w-clear_fluxerr_aper_3,f090w-clear_flag_aper_3,f090w-clear_bkg_aper_3,f090w-clear_mask_aper_3,f090w-clear_tot_corr,f105w_flux_aper_0,f105w_fluxerr_aper_0,f105w_flag_aper_0,f105w_bkg_aper_0,f105w_mask_aper_0,f105w_flux_aper_1,f105w_fluxerr_aper_1,f105w_flag_aper_1,f105w_bkg_aper_1,f105w_mask_aper_1,f105w_flux_aper_2,f105w_fluxerr_aper_2,f105w_flag_aper_2,f105w_bkg_aper_2,f105w_mask_aper_2,f105w_flux_aper_3,f105w_fluxerr_aper_3,f105w_flag_aper_3,f105w_bkg_aper_3,f105w_mask_aper_3,f105w_tot_corr,f110w_flux_aper_0,f110w_fluxerr_aper_0,f110w_flag_aper_0,f110w_bkg_aper_0,f110w_mask_aper_0,f110w_flux_aper_1,f110w_fluxerr_aper_1,f110w_flag_aper_1,f110w_bkg_aper_1,f110w_mask_aper_1,f110w_flux_aper_2,f110w_fluxerr_aper_2,f110w_flag_aper_2,f110w_bkg_aper_2,f110w_mask_aper_2,f110w_flux_aper_3,f110w_fluxerr_aper_3,f110w_flag_aper_3,f110w_bkg_aper_3,f110w_mask_aper_3,f110w_tot_corr,f115w-clear_flux_aper_0,f115w-clear_fluxerr_aper_0,f115w-clear_flag_aper_0,f115w-clear_bkg_aper_0,f115w-clear_mask_aper_0,f115w-clear_flux_aper_1,f115w-clear_fluxerr_aper_1,f115w-clear_flag_aper_1,f115w-clear_bkg_aper_1,f115w-clear_mask_aper_1,f115w-clear_flux_aper_2,f115w-clear_fluxerr_aper_2,f115w-clear_flag_aper_2,f115w-clear_bkg_aper_2,f115w-clear_mask_aper_2,f115w-clear_flux_aper_3,f115w-clear_fluxerr_aper_3,f115w-clear_flag_aper_3,f115w-clear_bkg_aper_3,f115w-clear_mask_aper_3,f115w-clear_tot_corr,f115wn-clear_flux_aper_0,f115wn-clear_fluxerr_aper_0,f115wn-clear_flag_aper_0,f115wn-clear_bkg_aper_0,f115wn-clear_mask_aper_0,f115wn-clear_flux_aper_1,f115wn-clear_fluxerr_aper_1,f115wn-clear_flag_aper_1,f115wn-clear_bkg_aper_1,f115wn-clear_mask_aper_1,f115wn-clear_flux_aper_2,f115wn-clear_fluxerr_aper_2,f115wn-clear_flag_aper_2,f115wn-clear_bkg_aper_2,f115wn-clear_mask_aper_2,f115wn-clear_flux_aper_3,f115wn-clear_fluxerr_aper_3,f115wn-clear_flag_aper_3,

In [35]:
# DEFINITIONS

rmax = 0.1  # spectrum selection target dist (arcsec)

keep = ["id", "z_spec", "ra", "dec"] # columns to keep

pattern = r"f.*_0" # regex pattern to match photometry columns

only_spec = True  # whether to only include sources with spec-z

RUN_NAME = "GOODS-S" # name for outputs
N_MIN_COLORS = 5  # minimum number of filters

USE_NIRCAM = True
USE_NIRISS = False # recommended
USE_WFC3 = True
USE_ACS = True

DROP_HST_IR = True  # Drop HST IR filters (e.g., F105W, F125W, F160W) to focus on JWST data

In [20]:
# --- Step 3: Cross-match photometric and spectroscopic catalogs ---

# Convert RA,Dec to 3D unit vectors
def radec_to_xyz(ra_deg, dec_deg):
    ra = np.radians(ra_deg)
    dec = np.radians(dec_deg)
    x = np.cos(dec) * np.cos(ra)
    y = np.cos(dec) * np.sin(ra)
    z = np.sin(dec)
    return np.vstack([x, y, z]).T

# Match photometric and spectroscopic catalogs using KDTree
def match_phot_spec(phot, spectra, rmax=1):
    """Match photometric and spectroscopic catalogs using KDTree.

    Parameters
    ----------
    phot : astropy Table
        Photometric catalog with 'ra' and 'dec' columns.
    spectra : astropy Table
        Spectroscopic catalog with 'ra', 'dec', and 'z_best' columns.
    rmax : float
        Maximum matching radius in arcseconds.

    Returns
    -------
    z_spec : np.ndarray
        Array of spectroscopic redshifts matched to photometric catalog.
        Unmatched entries are set to -1.
    """
    xyz_phot = radec_to_xyz(phot['ra'], phot['dec'])
    xyz_spec = radec_to_xyz(spectra['ra'], spectra['dec'])

    # Build KDTree of spectroscopic positions
    tree = cKDTree(xyz_spec)

    # Query nearest neighbor
    dist, idx = tree.query(xyz_phot, k=1)

    # dist is chord distance; convert to angular distance
    ang_dist_rad = 2 * np.arcsin(np.clip(dist/2, -1, 1))
    ang_dist_arcsec = np.degrees(ang_dist_rad) * 3600

    # Apply match threshold
    mask = ang_dist_arcsec < rmax

    # Build z_spec array
    z_spec = np.full(len(phot), -1.0)
    z_spec[mask] = spectra['z_best'][idx][mask]

    return z_spec

z_spec = match_phot_spec(phot, spectra, rmax=rmax)
phot['z_spec'] = z_spec

In [21]:
# --- Step 4: Get useful data out of photometric catalog ---

# Extract useful data columns and clean photometry
def extract_and_clean(phot, keep, pattern):
    # Extract filter data columns
    for col in phot.colnames:
        if re.match(pattern, col):
            keep.append(col)
    tab2 = phot[keep]

    # Recommended EAZY missing-data values
    BAD_FLUX = -99
    BAD_ERR  = 1e10

    print("Initial table size:", len(tab2))

    # Find relevant flux columns
    flux_cols = [c for c in tab2.colnames if "flux_" in c]
    err_cols  = [c for c in tab2.colnames if "fluxerr_" in c]
    flag_cols = [c for c in tab2.colnames if "flag_" in c]

    # Guarantee matching order
    flux_cols.sort()
    err_cols.sort()
    flag_cols.sort()

    # Loop through each band and mask out flagged data
    for fcol, ecol, flagcol in zip(flux_cols, err_cols, flag_cols):

        bad = tab2[flagcol] != 0            # flag ≠ 0 → bad measurement

        # Replace bad or NaN fluxes/errors with EAZY-missing
        tab2[fcol][bad] = BAD_FLUX
        tab2[ecol][bad] = BAD_ERR

        # Also catch NaNs
        nan = np.isnan(tab2[fcol]) | np.isnan(tab2[ecol])
        tab2[fcol][nan] = BAD_FLUX
        tab2[ecol][nan] = BAD_ERR

    # Remove rows with no usable data
    # a row is "good" if any band has (flux != BAD_FLUX) AND (error != BAD_ERR)
    has_data = np.zeros(len(tab2), dtype=bool)

    for fcol, ecol in zip(flux_cols, err_cols):
        valid = (tab2[fcol] != BAD_FLUX) & (tab2[ecol] != BAD_ERR)
        has_data |= valid

    # Filter the table
    tab3 = tab2[has_data]
    print("Finished! Final table size:", len(tab3))

    return tab3

result = extract_and_clean(phot, keep, pattern)

Initial table size: 52427
Finished! Final table size: 47400


In [36]:
# --- Get filters used ---

# Full list of JWST NIRCam + NIRISS + HST ACS + WFC3/UVIS filters
nircam_filters = ['f150w2', 'f322w2', 
                 'f079w', 'f090w', 'f115w', 'f150w', 'f200w', 'f277w', 'f356w', 'f444w',
                 'f140m', 'f162m', 'f182m', 'f210m', 'f250m', 'f300m', 'f335m', 'f360m', 'f410m', 'f430m', 'f460m', 'f480m',
                 'f164n', 'f187n', 'f212n', 'f323n', 'f405n', 'f466n', 'f470n']
niriss_filters = ['f090w', 'f115w', 'f150w', 'f200w', 'f277w', 'f356w', 'f444w',
                  'f140m', 'f158m', 'f380m', 'f430m', 'f480m']
acs_filters = ['f435w', 'f475w', 'f555w', 'f606w', 'f625w', 'f775w', 'f814w', 'f850lp']
wfc3_ir_filters = ['f105w', 'f110w', 'f125w', 'f140w', 'f160w',
                   'f098m', 'f127m', 'f139m', 'f153m', 
                   'f126n', 'f128n', 'f130n', 'f132n', 'f164n', 'f167n']
wfc3_uvis_filters = [ 
                     'f218w', 'f225w', 'f275w', 'f336w', 'f390w', 'f438w','f475w', 'f555w', 'f606w', 'f625w', 'f775w', 'f814w',
                     'f390m', 'f410m', 'f467m', 'f547m', 'f621m', 'f689m', 'f763m', 'f845m',
                     'f280n', 'f343n', 'f373n', 'f359n', 'f469n', 'f487n', 'f502n', 'f631n', 'f645n', 'f656n', 'f657n', 'f658n', 'f665n', 'f673n', 'f680n', 'f953n',
                     'fq378n', 'fq387n', 'fq437n', 'fq492n', 'fq508n', 'fq619n', 'fq674n', 'fq750n', 'fq889n', 'fq937n',
                     'f300x', 'f475x',
                     'f200lp', 'f350lp', 'f600lp', 'f850lp']
# clearp or wn -> NIRISS
# u -> WFC3/UVIS
# else match NIRCam & ACS

# 105w, 110w, 125w, 140w, 160w are actually WFC3/IR filters
def classify_filter(name):
    """
    Returns (instrument, canonical_filter_name) or ("unknown", name)
    according to your DAWN mapping rules.
    """
    f = name.lower()

    # --- Rule 1: NIRISS (clearp-... or "...n" narrow filters)
    if "clearp" in f or re.search(r"[a-z]n-", f):
        for flt in niriss_filters:
            if flt.lower() in f:
                return ("niriss", flt)
        return ("niriss-unmatched", name)

    # --- Rule 2: WFC3/UVIS (patterns like 606wu, 814wu, etc.)
    if re.search(r"\d[a-z][a-z]?u_", f):
        for flt in wfc3_uvis_filters:
            if flt.lower() in f:
                return ("wfc3-uvis", flt)
        return ("wfc3-uvis-unmatched", name)

    # --- Rule 3: Mis-labeled WFC3/IR filters in DAWN
    for yflt in ("105w","110w","125w","140w","160w"):
        if yflt in f:
            # map to canonical IR filter names
            return ("wfc3-ir", f"f{yflt}")

    # --- Rule 4: Otherwise try ACS, then NIRCam
    for flt in acs_filters:
        if flt.lower() in f:
            return ("acs", flt)

    for flt in nircam_filters:
        if flt.lower() in f:
            return ("nircam", flt)

    # --- Nothing matched
    return ("unknown", name)

# --- Categorize filters ---
tab_final = result[['id', 'z_spec']]
filter_cat = []
instrument_cat = []
for i, col in enumerate(result.columns):
    if re.match('.*flux.*', col):
        inst, flt = classify_filter(col)
        if inst == "unknown": # drop & log unknown inputs
            print(f"Unknown filter: {col}")
            continue
        if re.match('.*flux_.*', col): # avoid double-counting fluxerrs
            if inst == "nircam" and USE_NIRCAM:
                instrument_cat.append('nircam')
            elif inst == "niriss" and USE_NIRISS:
                instrument_cat.append('niriss')
            elif inst == "wfc3-uvis" and USE_WFC3:    
                instrument_cat.append('wfc3-uvis')
            elif inst == "wfc3-ir" and USE_WFC3 and not DROP_HST_IR:
                instrument_cat.append('wfc3-ir')
            elif inst == "acs" and USE_ACS:
                instrument_cat.append('acs')
            else:
                continue
            filter_cat.append(flt)
        tab_final[col] = result[col]

print(len(filter_cat), "filters classified.")
tab_final

Unknown filter: flux_aper_0
Unknown filter: fluxerr_aper_0
24 filters classified.


id,z_spec,f090w-clear_flux_aper_0,f090w-clear_fluxerr_aper_0,f105w_fluxerr_aper_0,f110w_fluxerr_aper_0,f115w-clear_flux_aper_0,f115w-clear_fluxerr_aper_0,f115wn-clear_fluxerr_aper_0,f125w_fluxerr_aper_0,f140w_fluxerr_aper_0,f150w-clear_flux_aper_0,f150w-clear_fluxerr_aper_0,f150wn-clear_fluxerr_aper_0,f160w_fluxerr_aper_0,f182m-clear_flux_aper_0,f182m-clear_fluxerr_aper_0,f200w-clear_flux_aper_0,f200w-clear_fluxerr_aper_0,f200wn-clear_fluxerr_aper_0,f210m-clear_flux_aper_0,f210m-clear_fluxerr_aper_0,f277w-clear_flux_aper_0,f277w-clear_fluxerr_aper_0,f335m-clear_flux_aper_0,f335m-clear_fluxerr_aper_0,f350lpu_flux_aper_0,f350lpu_fluxerr_aper_0,f356w-clear_flux_aper_0,f356w-clear_fluxerr_aper_0,f410m-clear_flux_aper_0,f410m-clear_fluxerr_aper_0,f430m-clear_flux_aper_0,f430m-clear_fluxerr_aper_0,f435w_flux_aper_0,f435w_fluxerr_aper_0,f444w-clear_flux_aper_0,f444w-clear_fluxerr_aper_0,f460m-clear_flux_aper_0,f460m-clear_fluxerr_aper_0,f475w_flux_aper_0,f475w_fluxerr_aper_0,f480m-clear_flux_aper_0,f480m-clear_fluxerr_aper_0,f606w_flux_aper_0,f606w_fluxerr_aper_0,f606wu_flux_aper_0,f606wu_fluxerr_aper_0,f775w_flux_aper_0,f775w_fluxerr_aper_0,f814w_flux_aper_0,f814w_fluxerr_aper_0,f814wu_flux_aper_0,f814wu_fluxerr_aper_0,f850lp_flux_aper_0,f850lp_fluxerr_aper_0,f850lpu_flux_aper_0,f850lpu_fluxerr_aper_0
,,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy
int32,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64
14,-1.0,-99.0,10000000000.0,0.0070530670977904455,0.0045469210929895455,-99.0,10000000000.0,10000000000.0,0.004355073762040078,0.015251490019497431,-99.0,10000000000.0,10000000000.0,0.005500861922387775,-99.0,10000000000.0,-99.0,10000000000.0,10000000000.0,-99.0,10000000000.0,-99.0,10000000000.0,-99.0,10000000000.0,0.03551747815483017,0.002652659458655895,-99.0,10000000000.0,-99.0,10000000000.0,-99.0,10000000000.0,0.02931640640105563,0.001852515688505181,-99.0,10000000000.0,-99.0,10000000000.0,-99.0,10000000000.0,-99.0,10000000000.0,0.03326438263264503,0.002229778166730082,-99.0,10000000000.0,0.05810943825952337,0.004958159510549983,0.0683601041209478,0.0018895175540949007,-99.0,10000000000.0,0.06149267188251269,0.0050998334570599956,-99.0,10000000000.0
17,-1.0,-99.0,10000000000.0,0.00781609291202605,0.004548990590550214,-99.0,10000000000.0,10000000000.0,0.004871904042293694,0.01513376166841917,-99.0,10000000000.0,10000000000.0,0.00587566141178991,-99.0,10000000000.0,-99.0,10000000000.0,10000000000.0,-99.0,10000000000.0,-99.0,10000000000.0,-99.0,10000000000.0,0.02582048354849763,0.002568308181892546,-99.0,10000000000.0,-99.0,10000000000.0,-99.0,10000000000.0,0.015866786978939215,0.001856448675506439,-99.0,10000000000.0,-99.0,10000000000.0,-99.0,10000000000.0,-99.0,10000000000.0,0.018234109248517386,0.0019168971645118758,-99.0,10000000000.0,0.05192521823661275,0.0049587403399534895,0.04766198941979042,0.0016354460459419228,-99.0,10000000000.0,0.046206845361964996,0.004619156056539485,-99.0,10000000000.0
18,-1.0,-99.0,10000000000.0,0.007831473210081695,0.004606372996969212,-99.0,10000000000.0,10000000000.0,0.005258152013098746,0.014945042852881686,-99.0,10000000000.0,10000000000.0,0.006504721298959082,-99.0,10000000000.0,-99.0,10000000000.0,10000000000.0,-99.0,10000000000.0,-99.0,10000000000.0,-99.0,10000000000.0,0.025175494628711083,0.0037150043991822745,-99.0,10000000000.0,-99.0,10000000000.0,-99.0,10000000000.0,0.00869517740091571,0.00189264152238509,-99.0,100

In [33]:
# -- Define filter loading function --

def load_filter_file(filepath):
    """
    Load throughput, remove commented lines, ensure 2 columns,
    and convert wavelengths (µm → Å) if necessary.
    """
    # Load ignoring comment lines
    data = np.genfromtxt(filepath, comments="#", skip_header=1)
    if data.ndim != 2 or data.shape[1] < 2:
        raise ValueError(f"Filter file {filepath} must contain 2+ columns")

    lam = data[:, 0]
    thr = data[:, 1]

    if lam[0] < 100:  # assume input is in µm
        # µm → Å
        lam *= 1e4
    
    else: # assume input is in Å
        pass

    return lam, thr

def sanitize_filter(lam, thr):
    """
    Sort by wavelength, remove non-finite values, remove duplicates,
    and enforce strictly increasing wavelength.
    """

    lam = np.array(lam, dtype=float)
    thr = np.array(thr, dtype=float)

    # Remove bad values
    ok = np.isfinite(lam) & np.isfinite(thr)
    lam = lam[ok]
    thr = thr[ok]

    if len(lam) == 0:
        return lam, thr

    # Sort
    order = np.argsort(lam)
    lam = lam[order]
    thr = thr[order]

    # Remove duplicated wavelengths
    unique_w, inv = np.unique(lam, return_inverse=True)
    if len(unique_w) < len(lam):
        # average throughput of duplicates
        new_thr = np.zeros_like(unique_w)
        for i in range(len(unique_w)):
            new_thr[i] = thr[inv == i].mean()
        lam, thr = unique_w, new_thr

    # Enforce strict monotonicity: tiny perturbation if needed
    dif = np.diff(lam)
    if np.any(dif <= 0):
        eps = 1e-6 * np.maximum(1.0, lam)
        for i in range(1, len(lam)):
            if lam[i] <= lam[i-1]:
                lam[i] = lam[i-1] + eps[i]

    return lam, thr

# -- Define run name and filter directories --
filter_dirs = {
    "nircam": Path("../throughputs/nircam_throughputs"),
    "niriss": Path("../throughputs/niriss_throughputs"),
    "wfc3-ir":   Path("../throughputs/wfc3-ir_throughputs"),
    "wfc3-uvis": Path("../throughputs/wfc3-uvis_throughputs"),
    "acs":    Path("../throughputs/acs_throughputs"),
}

# --- Step 5: write filters.res file ---
def write_filters(filter_dirs, instrument_cat, filter_cat):
    resolved_entries = []

    for instr, filt in zip(instrument_cat, filter_cat):
        this_dir = filter_dirs[instr]
        filter_files = list(this_dir.glob("*.txt"))

        # Find match for catalog filter name
        matches = [
            f for f in filter_files
            if f.name.lower().endswith(filt.lower() + ".txt")
        ]

        # Warn & skip for no match
        if not matches:
            print(f"WARNING: No file for {instr}:{filt}")
            resolved_entries.append((instr, filt, None))
            continue

        # Warn for multiple matches - uses first match
        if len(matches) > 1:
            print(f"WARNING: Multiple matches for {instr}:{filt} → {matches[0]}")

        resolved_entries.append((instr, filt, matches[0]))


    print("Resolved entries:", len(resolved_entries))

    # Write resolved filters to filters.res
    output_file = f"../inputs/{RUN_NAME}_filters.res"

    with open(output_file, "w") as fout:
        for instr, filt, path in resolved_entries:

            # Load .txt filter file (sanitized)
            lam, thr = load_filter_file(path)
            lam, thr = sanitize_filter(lam, thr)

            # Write to EAZY format
            fout.write(f"{len(lam):6d} {instr + '_' + filt}\n")

            for i, (L, T) in enumerate(zip(lam, thr), start=1):
                fout.write(f"{i:<3d}    {L:.5e} {T:.5e}\n")

    print(f"Wrote {output_file} with {len(resolved_entries)} filters.")

    return None

write_filters(filter_dirs, instrument_cat, filter_cat)

Resolved entries: 29
Wrote ../inputs/GOODS-S_filters.res with 29 filters.


In [24]:
tab_final[tab_final['z_spec'] != -1]

id,z_spec,f090w-clear_flux_aper_0,f090w-clear_fluxerr_aper_0,f105w_flux_aper_0,f105w_fluxerr_aper_0,f110w_flux_aper_0,f110w_fluxerr_aper_0,f115w-clear_flux_aper_0,f115w-clear_fluxerr_aper_0,f115wn-clear_flux_aper_0,f115wn-clear_fluxerr_aper_0,f125w_flux_aper_0,f125w_fluxerr_aper_0,f140w_flux_aper_0,f140w_fluxerr_aper_0,f150w-clear_flux_aper_0,f150w-clear_fluxerr_aper_0,f150wn-clear_flux_aper_0,f150wn-clear_fluxerr_aper_0,f160w_flux_aper_0,f160w_fluxerr_aper_0,f182m-clear_flux_aper_0,f182m-clear_fluxerr_aper_0,f200w-clear_flux_aper_0,f200w-clear_fluxerr_aper_0,f200wn-clear_flux_aper_0,f200wn-clear_fluxerr_aper_0,f210m-clear_flux_aper_0,f210m-clear_fluxerr_aper_0,f277w-clear_flux_aper_0,f277w-clear_fluxerr_aper_0,f335m-clear_flux_aper_0,f335m-clear_fluxerr_aper_0,f350lpu_flux_aper_0,f350lpu_fluxerr_aper_0,f356w-clear_flux_aper_0,f356w-clear_fluxerr_aper_0,f410m-clear_flux_aper_0,f410m-clear_fluxerr_aper_0,f430m-clear_flux_aper_0,f430m-clear_fluxerr_aper_0,f435w_flux_aper_0,f435w_fluxerr_aper_0,f444w-clear_flux_aper_0,f444w-clear_fluxerr_aper_0,f460m-clear_flux_aper_0,f460m-clear_fluxerr_aper_0,f475w_flux_aper_0,f475w_fluxerr_aper_0,f480m-clear_flux_aper_0,f480m-clear_fluxerr_aper_0,f606w_flux_aper_0,f606w_fluxerr_aper_0,f606wu_flux_aper_0,f606wu_fluxerr_aper_0,f775w_flux_aper_0,f775w_fluxerr_aper_0,f814w_flux_aper_0,f814w_fluxerr_aper_0,f814wu_flux_aper_0,f814wu_fluxerr_aper_0,f850lp_flux_aper_0,f850lp_fluxerr_aper_0,f850lpu_flux_aper_0,f850lpu_fluxerr_aper_0
,,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy,uJy
int32,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64
51,2.899657,-99.0,10000000000.0,0.05934750436201354,0.007747863975176004,-99.0,10000000000.0,-99.0,10000000000.0,-99.0,10000000000.0,0.05836482602745065,0.005813960837265127,0.07454173856541547,0.015297281653053711,-99.0,10000000000.0,-99.0,10000000000.0,0.07680881733071374,0.008678745699400121,0.155576006463941,0.004910301757365831,-99.0,10000000000.0,-99.0,10000000000.0,0.11869877883636511,0.005949957992984396,-99.0,10000000000.0,-99.0,10000000000.0,0.040481182047514316,0.003438643465467653,-99.0,10000000000.0,-99.0,10000000000.0,-99.0,10000000000.0,0.021217638421835407,0.0030755378791607543,-99.0,10000000000.0,-99.0,10000000000.0,-99.0,10000000000.0,-99.0,10000000000.0,0.055564281350179094,0.0019153045714293397,-99.0,10000000000.0,0.05753399916501058,0.004491354129896674,0.06709110077251543,0.0017313453755913638,-99.0,10000000000.0,0.06828083049252444,0.004686326233032491,-99.0,10000000000.0
70,3.0992495000000004,-99.0,10000000000.0,0.013547859142576911,0.007710653568530049,0.016397143473306036,0.004596722908305905,-99.0,10000000000.0,-99.0,10000000000.0,0.022680654726060102,0.004125743593960061,0.03173411913046979,0.01544645504367884,-99.0,10000000000.0,-99.0,10000000000.0,0.023161819333335592,0.0051060588508098,-99.0,10000000000.0,-99.0,10000000000.0,-99.0,10000000000.0,-99.0,10000000000.0,-99.0,10000000000.0,-99.0,10000000000.0,0.019411597498366,0.00287906276960713,-99.0,10000000000.0,-99.0,10000000000.0,-99.0,10000000000.0,0.013094014139735275,0.0017721716063736344,-99.0,10000000000.0,-99.0,10000000000.0,-99.0,10000000000.0,-99.0,10000000000.0,0.025145518528217576,0.0023258259971563134,-99.0,10000000000.0,0.02807122202646021,0.005344385325589873,0.03181739864321733,0.0015660068096469187,-99.0,10000

In [25]:
# --- Step 7: Write EAZY-ready table ---
def write_photometry(result, instrument_cat, filter_cat, only_spec=False):
    if only_spec == True:
        tab_final = result[result['z_spec'] != -1]
    else:
        tab_final = result

    pretty_header = "# " + " ".join(["id"] + [f"{x}_{instr + '_' + filt}" for instr, filt in zip(instrument_cat, filter_cat) for x in ("f","e")] + ['z_spec'])
    machine_header = "# " + " ".join(["id"] + [f"{x}{i+1}" for i in range(len(filter_cat)) for x in ("F","E")] + ['z_spec'])

    n_obj = len(tab_final)
    n_filters = len(filter_cat)

    # Pre-format ID and z columns
    id_col = [f"{int(x):6d}" for x in tab_final['id']]
    z_col  = [f"{x: .4f}" for x in tab_final['z_spec']]

    photo_cols = []

    filter_columns = [c for c in tab_final.columns 
                    if ("flux_") in c or ("fluxerr_") in c]

    for c in filter_columns:
        flux = tab_final[c]
        photo_cols.append([f"{x: 6e}" for x in flux])

    # Transpose to per-row layout
    rows = zip(
        id_col,
        *(photo_cols),
        z_col,
    )

    with open(f"../inputs/{RUN_NAME}_photometry.cat", "w", encoding="ascii") as fp:
        fp.write(pretty_header + "\n")
        fp.write(machine_header + "\n")

        for fields in rows:
            fp.write(" ".join(fields) + "\n")

    print(f"Wrote photometry.cat with {n_obj} objects and {n_filters} filters.")

    # --- Step 8: Write zphot.translate file ---
    with open('../inputs/zphot.translate', "w") as f:
        n = 1
        for instr, filt in zip(instrument_cat, filter_cat):
            for x in ("f","e"):
                f.write(f"{x}_{instr + '_' + filt} {x.upper()}{n} \n")
            n += 1

    print(f"Successfully wrote zphot.translate with {n_filters} filters.")

    return None

write_photometry(tab_final, instrument_cat, filter_cat, only_spec=only_spec)

Wrote photometry.cat with 1792 objects and 32 filters.
Successfully wrote zphot.translate with 32 filters.


In [14]:
tab_final = result[result['z_spec'] != -1][0:1772]
tab_final

id,z_spec,ra,dec,flux_aper_0,fluxerr_aper_0,flag_aper_0,f090w-clear_flux_aper_0,f090w-clear_fluxerr_aper_0,f090w-clear_flag_aper_0,f090w-clear_bkg_aper_0,f090w-clear_mask_aper_0,f105w_flux_aper_0,f105w_fluxerr_aper_0,f105w_flag_aper_0,f105w_bkg_aper_0,f105w_mask_aper_0,f110w_flux_aper_0,f110w_fluxerr_aper_0,f110w_flag_aper_0,f110w_bkg_aper_0,f110w_mask_aper_0,f115w-clear_flux_aper_0,f115w-clear_fluxerr_aper_0,f115w-clear_flag_aper_0,f115w-clear_bkg_aper_0,f115w-clear_mask_aper_0,f115wn-clear_flux_aper_0,f115wn-clear_fluxerr_aper_0,f115wn-clear_flag_aper_0,f115wn-clear_bkg_aper_0,f115wn-clear_mask_aper_0,f125w_flux_aper_0,f125w_fluxerr_aper_0,f125w_flag_aper_0,f125w_bkg_aper_0,f125w_mask_aper_0,f140w_flux_aper_0,f140w_fluxerr_aper_0,f140w_flag_aper_0,f140w_bkg_aper_0,f140w_mask_aper_0,f150w-clear_flux_aper_0,f150w-clear_fluxerr_aper_0,f150w-clear_flag_aper_0,f150w-clear_bkg_aper_0,f150w-clear_mask_aper_0,f150wn-clear_flux_aper_0,f150wn-clear_fluxerr_aper_0,f150wn-clear_flag_aper_0,f150wn-clear_bkg_aper_0,f150wn-clear_mask_aper_0,f160w_flux_aper_0,f160w_fluxerr_aper_0,f160w_flag_aper_0,f160w_bkg_aper_0,f160w_mask_aper_0,f182m-clear_flux_aper_0,f182m-clear_fluxerr_aper_0,f182m-clear_flag_aper_0,f182m-clear_bkg_aper_0,f182m-clear_mask_aper_0,f200w-clear_flux_aper_0,f200w-clear_fluxerr_aper_0,f200w-clear_flag_aper_0,f200w-clear_bkg_aper_0,f200w-clear_mask_aper_0,f200wn-clear_flux_aper_0,f200wn-clear_fluxerr_aper_0,f200wn-clear_flag_aper_0,f200wn-clear_bkg_aper_0,f200wn-clear_mask_aper_0,f210m-clear_flux_aper_0,f210m-clear_fluxerr_aper_0,f210m-clear_flag_aper_0,f210m-clear_bkg_aper_0,f210m-clear_mask_aper_0,f277w-clear_flux_aper_0,f277w-clear_fluxerr_aper_0,f277w-clear_flag_aper_0,f277w-clear_bkg_aper_0,f277w-clear_mask_aper_0,f335m-clear_flux_aper_0,f335m-clear_fluxerr_aper_0,f335m-clear_flag_aper_0,f335m-clear_bkg_aper_0,f335m-clear_mask_aper_0,f350lpu_flux_aper_0,f350lpu_fluxerr_aper_0,f350lpu_flag_aper_0,f350lpu_bkg_aper_0,f350lpu_mask_aper_0,f356w-clear_flux_aper_0,f356w-clear_fluxerr_aper_0,f356w-clear_flag_aper_0,f356w-clear_bkg_aper_0,f356w-clear_mask_aper_0,f410m-clear_flux_aper_0,f410m-clear_fluxerr_aper_0,f410m-clear_flag_aper_0,f410m-clear_bkg_aper_0,f410m-clear_mask_aper_0,f430m-clear_flux_aper_0,f430m-clear_fluxerr_aper_0,f430m-clear_flag_aper_0,f430m-clear_bkg_aper_0,f430m-clear_mask_aper_0,f435w_flux_aper_0,f435w_fluxerr_aper_0,f435w_flag_aper_0,f435w_bkg_aper_0,f435w_mask_aper_0,f444w-clear_flux_aper_0,f444w-clear_fluxerr_aper_0,f444w-clear_flag_aper_0,f444w-clear_bkg_aper_0,f444w-clear_mask_aper_0,f460m-clear_flux_aper_0,f460m-clear_fluxerr_aper_0,f460m-clear_flag_aper_0,f460m-clear_bkg_aper_0,f460m-clear_mask_aper_0,f475w_flux_aper_0,f475w_fluxerr_aper_0,f475w_flag_aper_0,f475w_bkg_aper_0,f475w_mask_aper_0,f480m-clear_flux_aper_0,f480m-clear_fluxerr_aper_0,f480m-clear_flag_aper_0,f480m-clear_bkg_aper_0,f480m-clear_mask_aper_0,f606w_flux_aper_0,f606w_fluxerr_aper_0,f606w_flag_aper_0,f606w_bkg_aper_0,f606w_mask_aper_0,f606wu_flux_aper_0,f606wu_fluxerr_aper_0,f606wu_flag_aper_0,f606wu_bkg_aper_0,f606wu_mask_aper_0,f775w_flux_aper_0,f775w_fluxerr_aper_0,f775w_flag_aper_0,f775w_bkg_aper_0,f775w_mask_aper_0,f814w_flux_aper_0,f814w_fluxerr_aper_0,f814w_flag_aper_0,f814w_bkg_aper_0,f814w_mask_aper_0,f814wu_flux_aper_0,f814wu_fluxerr_aper_0,f814wu_flag_aper_0,f814wu_bkg_aper_0,f814wu_mask_aper_0,f850lp_flux_aper_0,f850lp_fluxerr_aper_0,f850lp_flag_aper_0,f850lp_bkg_aper_0,f850lp_mask_aper_0,f850lpu_flux_aper_0,f850lpu_fluxerr_aper_0,f850lpu_flag_aper_0,f850lpu_bkg_aper_0,f850lpu_mask_aper_0
,,deg,deg,uJy,uJy,,uJy,uJy,,uJy,,uJy,uJy,,uJy,,uJy,uJy,,uJy,,uJy,uJy,,uJy,,uJy,uJy,,uJy,,uJy,uJy,,uJy,,uJy,uJy,,uJy,,uJy,uJy,,uJy,,uJy,uJy,,uJy,,uJy,uJy,,uJy,,uJy,uJy,,uJy,,uJy,uJy,,uJy,,uJy,uJy,,uJy,,uJy,uJy,,uJy,,uJy,uJy,,uJy,,uJy,uJy,,uJy,,uJy,uJy,,uJy,,uJy,uJy,,uJy,,uJy,uJy,,uJy,,uJy,uJy,,uJy,,uJy,uJy,,uJy,,uJy,uJy,,uJy,,uJy,uJy,,uJy,,uJy,uJy,,uJy,,uJy,uJy,,uJy,,uJy,uJy,,uJy,,uJy,uJy,,uJy,,uJy,uJy,,uJy,,uJy,uJy,,uJy,,uJy,uJy,,uJy,,u

In [12]:
# TESTING - Diagnosing truncated EAZY output

pretty_header = "# " + " ".join(["id"] + [f"{x}_{instr + '_' + filt}" for instr, filt in zip(instrument_cat, filter_cat) for x in ("f","e")] + ['z_spec'])
machine_header = "# " + " ".join(["id"] + [f"{x}{i+1}" for i in range(len(filter_cat)) for x in ("F","E")] + ['z_spec'])

n_obj = len(tab_final)
n_filters = len(filter_cat)

# Pre-format ID and z columns
id_col = [f"{int(x):6d}" for x in tab_final['id']]
z_col  = [f"{x: .4f}" for x in tab_final['z_spec']]

photo_cols = []

filter_columns = [c for c in tab_final.columns 
                if ("flux_") in c or ("fluxerr_") in c]

for c in filter_columns:
    flux = tab_final[c]
    photo_cols.append([f"{x: 6e}" for x in flux])

# Transpose to per-row layout
rows = zip(
    id_col,
    *(photo_cols),
    z_col,
)

with open(f"../inputs/{RUN_NAME}_photometry.cat", "w", encoding="ascii") as fp:
    fp.write(pretty_header + "\n")
    fp.write(machine_header + "\n")

    for fields in rows:
        fp.write(" ".join(fields) + "\n")

print(f"Wrote photometry.cat with {n_obj} objects and {n_filters} filters.")

# --- Step 8: Write zphot.translate file ---
with open('../inputs/zphot.translate', "w") as f:
    n = 1
    for instr, filt in zip(instrument_cat, filter_cat):
        for x in ("f","e"):
            f.write(f"{x}_{instr + '_' + filt} {x.upper()}{n} \n")
        n += 1

print(f"Successfully wrote zphot.translate with {n_filters} filters.")

Wrote photometry.cat with 1772 objects and 32 filters.
Successfully wrote zphot.translate with 32 filters.


In [26]:
# --- Step 9: Write final EAZY input ---

def write_eazy_param(outfile="eazy_full.param", params=None):
    """
    Create an EAZY .param file from a dictionary.
    """

    if params is None:
        raise ValueError("You must supply a params dictionary.")

    with open(outfile, "w") as f:

        # Header
        f.write("# EAZY parameter file\n")
        f.write(f"# Auto-generated on {datetime.now()} for catalog {RUN_NAME} \n\n")

        # Write each parameter line
        for key, value in params.items():
            if isinstance(value, list):
                value = " ".join(map(str, value))
            line = f"{key:<25} {value}"
            f.write(line + "\n")

    print(f"✔️ Wrote {outfile}")

# --- For JWST GOODS/CEERS ---
params = {
    ## Filters
    'FILTERS_RES':       f"{RUN_NAME}_filters.res",  # Filter transmission data
    'FILTER_FORMAT':     1,                  # Format of FILTERS_RES file -- 0: energy-  1: photon-counting detector
    'SMOOTH_FILTERS':    'n',                  # Smooth filter curves with Gaussian
    'SMOOTH_SIGMA':      100.,             # Gaussian sigma (in Angstroms) to smooth filters

    ## Templates
    'TEMPLATES_FILE':       'templates/fsps_full/fsps_QSF_12_v3.param', # FSPS Full template
    'TEMPLATE_COMBOS':      'a',                 # Template combination options: 
    'NMF_TOLERANCE':        1.e-04,           # Tolerance for non-negative combinations (TEMPLATE_COMBOS=a)
    'WAVELENGTH_FILE':      'templates/uvista_nmf/lambda.def', # Wavelength grid definition file
    'TEMP_ERR_FILE':        'templates/template_error_cosmos2020.txt', # Template error definition file
    'TEMP_ERR_A2':          0.50,              # Template error amplitude
    'SYS_ERR':              0.00,              # Systematic flux error (% of flux)
    'APPLY_IGM':            'y',                    # Apply IGM absorption (1/y=Inoue2014, 2/x=Madau1995)
    'LAF_FILE':             'templates/LAFcoeff.txt', # File containing the Lyman alpha forest data from Inoue(2014)
    'DLA_FILE':             'templates/DLAcoeff.txt', # File containing the damped Lyman absorber data from Inoue(2014)
    'SCALE_2175_BUMP':      0.00,              # Scaling of 2175A bump.  Values 0.13 (0.27) absorb ~10 (20) % at peak.

    'DUMP_TEMPLATE_CACHE':  'n',                  # Write binary template cache
    'USE_TEMPLATE_CACHE':   'n',                  # Load in template cache
    'CACHE_FILE':           'photz.tempfilt',     # Template cache file (in OUTPUT_DIRECTORY)

    ## Input Files
    'CATALOG_FILE':         f"{RUN_NAME}_photometry.cat", # Catalog data file
    'MAGNITUDES':           'n',                  # Catalog photometry in magnitudes rather than f_nu fluxes
    'NOT_OBS_THRESHOLD':    -90,            # Ignore flux point if <NOT_OBS_THRESH
    'N_MIN_COLORS':         N_MIN_COLORS,                  # Require N_MIN_COLORS to fit

    ## Output Files
    'OUTPUT_DIRECTORY':     'OUTPUT', # Directory to put output files in
    'MAIN_OUTPUT_FILE':     'photz',            # Main output file, .zout
    'PRINT_ERRORS':         'y',                   # Print 68, 95 and 99% confidence intervals
    'CHI2_SCALE':           1.0,               # Scale ML Chi-squared values to improve confidence intervals
    'VERBOSE_LOG':          'y',                   # Dump information from the run into [MAIN_OUTPUT_FILE].param
    'OBS_SED_FILE':         'n',                   # Write out observed SED/object, .obs_sed
    'TEMP_SED_FILE':        'n',                   # Write out best template fit/object, .temp_sed
    'POFZ_FILE':            'n',                   # Write out Pofz/object, .pz
    'BINARY_OUTPUT':        'y',                   # Save OBS_SED, TEMP_SED, PZ in binary format to read with e.g IDL

    ## Redshift / Mag prior
    'APPLY_PRIOR':          'n',                   # Apply apparent magnitude prior
    'PRIOR_FILE':           'templates/prior_K_extend.dat', # File containing prior grid
    'PRIOR_FILTER':         1,                   # Filter from FILTER_RES corresponding to the columns in PRIOR_FILE
    'PRIOR_ABZP':           25.0,              # AB zeropoint of fluxes in catalog.  Needed for calculating apparent mags!

    ## Redshift Grid
    'FIX_ZSPEC':            'n',                  # Fix redshift to catalog zspec
    'Z_MIN':                0.01,              # Minimum redshift
    'Z_MAX':                20.0,               # Maximum redshift
    'Z_STEP':               0.01,              # Redshift step size
    'Z_STEP_TYPE':          1,                    #  0 = ZSTEP, 1 = Z_STEP*(1+z)

    ## Zeropoint Offsets
    'GET_ZP_OFFSETS':       'n',                  # Look for zphot.zeropoint file and compute zeropoint offsets
    'ZP_OFFSET_TOL':        1.000e-04,          # Tolerance for iterative fit for zeropoint offsets [not implemented]

    ## Rest-frame colors
    'REST_FILTERS':         '---',              # Comma-separated list of rest frame filters to compute
    'RF_PADDING':           1000,               # Padding (Ang) for choosing observed filters around specified rest-frame pair.
    'RF_ERRORS':            'n',                   # Compute RF color errors from p(z)
    'Z_COLUMN':             'z_peak',           # Redshift to use for rest-frame color calculation (z_a, z_p, z_m1, z_m2, z_peak)
    'USE_ZSPEC_FOR_REST':   'y',                   # Use z_spec when available for rest-frame colors
    'READ_ZBIN':            'n',                # Get redshifts from OUTPUT_DIRECTORY/MAIN_OUTPUT_FILE.zbin rather than fitting them.

    ## Cosmology
    'H0':                   70.0,             # Hubble constant (km/s/Mpc)
    'OMEGA_M':              0.3,              # Omega_matter
    'OMEGA_L':              0.7              # Omega_lambda
}


write_eazy_param(f"../inputs/{RUN_NAME}_eazy_full.param", params)

✔️ Wrote ../inputs/GOODS-S_eazy_full.param
